# Data Cleaning Notebook

Week 1-2 stuff

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
np.random.seed(42)


In [ ]:
SRC_PATH = Path.cwd() / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from project_utils import (
    EXPECTED_AUDIO_COLS,
    clean_spotify_dataframe,
    ensure_dir,
    get_numeric_model_frame,
    load_spotify_dataframe,
    split_data,
)


In [ ]:
raw_df, source_path = load_spotify_dataframe(data_dir="data", allow_download=True)
print(f"Loaded dataset from: {source_path}")
print(f"Raw shape: {raw_df.shape}")
raw_df.head()


In [ ]:
print("Raw columns:")
print(raw_df.columns.tolist())

missing_report = raw_df.isna().mean().sort_values(ascending=False).head(15)
print()
print("Top missing-value rates:")
print((missing_report * 100).round(2).astype(str) + "%")


In [ ]:
clean_df = clean_spotify_dataframe(raw_df)
model_df = get_numeric_model_frame(clean_df)

print(f"Rows before cleaning: {len(raw_df):,}")
print(f"Rows after cleaning:  {len(clean_df):,}")
print(f"Numeric modeling columns: {len(model_df.columns)}")

proposal_cols_available = [c for c in EXPECTED_AUDIO_COLS if c in clean_df.columns]
print()
print("Proposal-aligned columns available:")
print(proposal_cols_available)

clean_df.head()


In [ ]:
train_df, val_df, test_df = split_data(model_df, test_size=0.15, val_size=0.15, random_state=42)

processed_dir = ensure_dir("data/processed")
clean_path = processed_dir / "spotify_clean.csv"
train_path = processed_dir / "train.csv"
val_path = processed_dir / "val.csv"
test_path = processed_dir / "test.csv"

clean_df.to_csv(clean_path, index=False)
train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Saved: {clean_path}")
print(f"Saved: {train_path}")
print(f"Saved: {val_path}")
print(f"Saved: {test_path}")
print()
print(f"Split sizes -> train: {len(train_df):,}, val: {len(val_df):,}, test: {len(test_df):,}")


In [ ]:
plot_cols = [
    c for c in ["danceability", "energy", "valence", "acousticness", "loudness", "tempo"]
    if c in model_df.columns
]

if plot_cols:
    n_cols = 3
    n_rows = int(np.ceil(len(plot_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
    axes = np.array(axes).reshape(-1)

    for i, col in enumerate(plot_cols):
        axes[i].hist(model_df[col], bins=30)
        axes[i].set_title(col)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Feature Distributions (Cleaned Data)", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No proposal columns found for plotting.")
